# Reprodução — supervised_detection_some_ip (Alkhatib, LSTM binário) na T4

Treina/avalia o **LSTM binário** (58 features, janela 128) do repositório
[Alkhatibnatasha/supervised_detection_some_ip](https://github.com/Alkhatibnatasha/supervised_detection_some_ip)
numa **GPU T4** do Colab. O notebook é autossuficiente: clona o repo, **aplica os ajustes**
que descobrimos para fazê-lo rodar, baixa o dataset, organiza em `train/valid/test` e roda.

**Ajustes embutidos (sem eles o repo não roda):**
1. `output_dir` absoluto (o dataloader faz `os.chdir`, que quebra caminhos relativos);
2. organizar os pickles (4 pastas por ataque) em `train/valid/test`;
3. modo de avaliação é `predict` (o README diz `test`);
4. `stride` moderado (16) — `stride=1` gera centenas de milhares de janelas e estoura a RAM.

> **Nota honesta:** isto reproduz o modelo **binário deles nos dados deles**. **Não** habilita
> testá-lo no nosso tráfego gerado — para isso faltaria o extrator cru→58 features, que o repo
> não publica.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SEM GPU — ative Runtime>Change runtime type>T4')

In [ ]:
# 1) Clona o repositório original
%cd /content
!rm -rf supervised_detection_some_ip
!git clone -q https://github.com/Alkhatibnatasha/supervised_detection_some_ip
print('clonado')

In [ ]:
# 2) Aplica os ajustes no someip_lstm.py (caminho absoluto + stride moderado)
f = '/content/supervised_detection_some_ip/network_configuration_1/someip_lstm.py'
s = open(f).read()
s = s.replace('options["output_dir"] = "../output/network_configuration_1/"',
              'options["output_dir"] = os.path.abspath(os.path.join(cwd_orig, "../output/network_configuration_1/")) + "/"')
s = s.replace('options["stride"] = 1', 'options["stride"] = 16')
open(f, 'w').write(s)
print('patches aplicados:', 'output_dir absoluto' , '| stride=16')

In [ ]:
# 3) Baixa o dataset do Dropbox e descompacta
%cd /content
!wget -q -O dataset.zip "https://www.dropbox.com/sh/k5jnnplxb7ptw5b/AAAbS0N5RkVazV-7DG1MvEaZa?dl=1"
!rm -rf raw_data && mkdir -p raw_data && (cd raw_data && unzip -q ../dataset.zip)
import glob
print('amostras encontradas:', len(glob.glob('/content/raw_data/**/*_x.pickle', recursive=True)))

In [ ]:
# 4) Organiza os pickles em train/valid/test (70/15/15)
import os, glob, shutil
REPO = '/content/supervised_detection_some_ip'
dst = f'{REPO}/output/network_configuration_1/data'
for sp in ('train', 'valid', 'test'):
    os.makedirs(f'{dst}/{sp}', exist_ok=True)
xs = sorted(glob.glob('/content/raw_data/**/*_x.pickle', recursive=True))
moved = {'train': 0, 'valid': 0, 'test': 0}
for i, xp in enumerate(xs):
    yp = xp.replace('_x.pickle', '_y.pickle')
    if not os.path.exists(yp):
        continue
    sp = 'train' if i % 5 < 3 else ('valid' if i % 5 == 3 else 'test')
    for p in (xp, yp):
        shutil.copy(p, f'{dst}/{sp}/{os.path.basename(p)}')
    moved[sp] += 1
print('pares por split:', moved)

In [ ]:
# 5) Treina o LSTM (usa a GPU automaticamente). Early-stop em 10 épocas sem melhora.
%cd /content/supervised_detection_some_ip/network_configuration_1
!python someip_lstm.py train

In [ ]:
# 6) Avalia (predict) — varre limiares e reporta F1 / Recall / Precision
!python someip_lstm.py predict

## Notas

- **GPU acelera o treino**; o gargalo de RAM era o *windowing* com `stride=1` (resolvido com
  `stride=16`).
- Outros modelos do repo: troque `someip_lstm.py` por `someip_rnn.py`, `someip_transformer.py`
  ou `someip_mlp.py` (mesmos ajustes de caminho/stride podem ser necessários).
- **Escopo:** reprodução do detector **binário** do Alkhatib **nos dados dele**. Não substitui o
  Experimento A no nosso tráfego gerado (faltaria o extrator cru→features, não publicado).